# Job Route Classifier (Colab)

This notebook trains a small classifier for three labels used by the job-automation pipeline:

- `easy_apply`
- `offsite`
- `manual_review_candidate`

The workflow is:

1. clone or mount this repo in Colab
2. export a labeled dataset from the SQLite job DB
3. train a fast TF-IDF baseline
4. fine-tune a small transformer (`prajjwal1/bert-tiny`) as the first SLM checkpoint
5. save the model artifacts for later integration behind a feature flag


In [ ]:
# Optional: mount Drive if your repo or sqlite db lives there.
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
REPO_DIR = '/content/Job-Ops-Console'  # change if needed
SQLITE_DB = f'{REPO_DIR}/input/crawled_job/linkedin_jobs_jd.sqlite'
OUT_DIR = f'{REPO_DIR}/artifacts/ml/job_route'


In [ ]:
!pip -q install pandas scikit-learn datasets transformers evaluate accelerate

In [ ]:
!python {REPO_DIR}/scripts/python/export_job_route_dataset.py --sqlite-db {SQLITE_DB} --out-dir {OUT_DIR}

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path(f'{OUT_DIR}/job_route_dataset_summary.json').read_text(encoding='utf-8'))
summary

In [ ]:
import pandas as pd

df = pd.read_csv(f'{OUT_DIR}/job_route_dataset.csv')
df[['label', 'split']].value_counts().sort_index()

## 1. Fast Baseline

This gives a cheap sanity check before spending Colab GPU time.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'val'].copy()
test_df = df[df['split'] == 'test'].copy()

baseline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=40000, ngram_range=(1, 2), min_df=2)),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
])
baseline.fit(train_df['text'], train_df['label'])
val_pred = baseline.predict(val_df['text'])
print(classification_report(val_df['label'], val_pred, digits=4))

## 2. Small Transformer / SLM

This is the first real SLM checkpoint. `bert-tiny` is intentionally small so you can iterate quickly on Colab.

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
import evaluate
import numpy as np

label_list = sorted(df['label'].unique().tolist())
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

def frame_to_dataset(frame):
    data = frame[['text', 'label']].copy()
    data['label_id'] = data['label'].map(label2id)
    return Dataset.from_pandas(data[['text', 'label_id']], preserve_index=False)

dataset = DatasetDict({
    'train': frame_to_dataset(train_df),
    'validation': frame_to_dataset(val_df),
    'test': frame_to_dataset(test_df),
})

model_name = 'prajjwal1/bert-tiny'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256)

tokenized = dataset.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
metric_f1 = evaluate.load('f1')
metric_acc = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = metric_f1.compute(predictions=preds, references=labels, average='macro')['f1']
    acc = metric_acc.compute(predictions=preds, references=labels)['accuracy']
    return {'accuracy': acc, 'macro_f1': f1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir=f'{OUT_DIR}/bert_tiny_runs',
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    report_to='none',
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
test_metrics = trainer.evaluate(tokenized['test'])
test_metrics

In [ ]:
export_dir = Path(f'{OUT_DIR}/bert_tiny_export')
export_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(export_dir))
tokenizer.save_pretrained(str(export_dir))
Path(export_dir / 'label_map.json').write_text(json.dumps(label2id, ensure_ascii=False, indent=2), encoding='utf-8')
print(export_dir)

## 3. Next Step Back In Repo

After training, keep integration low-risk:

- keep current heuristic routing as the default path
- add a feature flag for classifier inference
- compare classifier output vs current route decisions offline first
- only promote to live routing after measuring precision for `easy_apply` and recall for `manual_review_candidate`
